In [ ]:
# Example 1: basic connect + send via dispatcher.
from pydispatch import dispatcher

received = []

def handler(sender, **kwargs):
    received.append((sender, kwargs))

SIGNAL = 'test-signal-1'
dispatcher.connect(handler, signal=SIGNAL, sender=dispatcher.Any)
dispatcher.send(signal=SIGNAL, sender='sender-a', payload=42)

print(f"received: {received}")
assert len(received) == 1
assert received[0][0] == 'sender-a'
assert received[0][1]['payload'] == 42


In [ ]:
# Example 2: disconnect stops further delivery.
from pydispatch import dispatcher

count = [0]

def listener(sender, **kwargs):
    count[0] += 1

SIGNAL = 'test-signal-2'
dispatcher.connect(listener, signal=SIGNAL, sender=dispatcher.Any)
dispatcher.send(signal=SIGNAL, sender='s1')
dispatcher.send(signal=SIGNAL, sender='s2')
assert count[0] == 2

dispatcher.disconnect(listener, signal=SIGNAL, sender=dispatcher.Any)
dispatcher.send(signal=SIGNAL, sender='s3')

print(f"final count after disconnect: {count[0]}")
assert count[0] == 2


In [ ]:
# Example 3: sender-specific routing — only matching sender fires.
from pydispatch import dispatcher

calls = []

def specific_handler(sender, **kwargs):
    calls.append(sender)

SIGNAL = 'test-signal-3'
SENDER = object()
OTHER = object()

dispatcher.connect(specific_handler, signal=SIGNAL, sender=SENDER)
dispatcher.send(signal=SIGNAL, sender=SENDER, msg='match')
dispatcher.send(signal=SIGNAL, sender=OTHER, msg='no-match')

print(f"calls: {len(calls)}")
assert len(calls) == 1
assert calls[0] is SENDER


In [ ]:
# Example 4: getReceivers returns connected listeners.
from pydispatch import dispatcher

def a(sender, **kwargs):
    pass

def b(sender, **kwargs):
    pass

SIGNAL = 'test-signal-4'
dispatcher.connect(a, signal=SIGNAL, sender=dispatcher.Any)
dispatcher.connect(b, signal=SIGNAL, sender=dispatcher.Any)

receivers = list(dispatcher.getReceivers(sender=dispatcher.Any, signal=SIGNAL))
print(f"receiver count: {len(receivers)}")
assert len(receivers) >= 2


In [ ]:
# Example 5: robust.sendRobust catches receiver exceptions and returns them.
from pydispatch import dispatcher, robust

def good(sender, **kwargs):
    return 'ok'

def bad(sender, **kwargs):
    raise RuntimeError('boom')

SIGNAL = 'test-signal-5'
dispatcher.connect(good, signal=SIGNAL, sender=dispatcher.Any)
dispatcher.connect(bad, signal=SIGNAL, sender=dispatcher.Any)

results = robust.sendRobust(signal=SIGNAL, sender='x')
outcomes = [type(r[1]).__name__ if isinstance(r[1], Exception) else r[1] for r in results]

print(f"outcomes: {outcomes}")
assert 'ok' in outcomes
assert 'RuntimeError' in outcomes
